# Notebook 02 - Preprocessing dan Pelabelan Awal

**Peneliti:** Khoeru Roziqin (24060119120031)  
**Skripsi:** Analisis Sentimen Opini Publik di Platform Media Sosial X Mengenai Program Makan Bergizi Gratis Menggunakan BERT-CNN

Menjalankan pipeline preprocessing 4-tahap (case folding, cleaning, tokenization, normalization) menghasilkan `text_bert`, kemudian melakukan pelabelan otomatis dengan IndoRoBERTa sebagai pseudo-annotator untuk perhitungan Cohen's Kappa terhadap anotasi manual.


## 1. Setup dan Konfigurasi

Notebook dijalankan pada Kaggle Notebook dengan GPU aktif. Direktori output mengikuti struktur repositori lokal.


In [ ]:
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import cohen_kappa_score, confusion_matrix

warnings.filterwarnings("ignore")
tqdm.pandas()

assert os.path.exists("/kaggle/working"), "Notebook ini dirancang untuk Kaggle Notebook."

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"PyTorch      : {torch.__version__}")
print(f"GPU tersedia : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device   : {torch.cuda.get_device_name(0)}")


In [ ]:
# -- Path (mirror struktur repositori lokal) -----------------------------------
KAGGLE_INPUT = "/kaggle/input"
WORK_DIR     = "/kaggle/working"

INPUT_PATH   = os.path.join(KAGGLE_INPUT, "mbg-labeled", "raw_sample_labeled_mbg.csv")
PATH_KAMUS   = os.path.join(KAGGLE_INPUT, "mbg-kamus")

FINAL_DIR    = os.path.join(WORK_DIR, "dataset", "final")
RESULTS_DIR  = os.path.join(WORK_DIR, "research", "results", "02_preprocessing_labeling")
for d in (FINAL_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

VALIDATED_PATH = os.path.join(FINAL_DIR, "final_validated_mbg.csv")   # full audit trail
LABELED_PATH   = os.path.join(FINAL_DIR, "final_mbg_labeled.csv")     # input untuk NB03

# -- Parameter -----------------------------------------------------------------
RANDOM_SEED    = 42
BATCH_SIZE     = 32
MAX_LENGTH     = 128
MIN_WORD_COUNT = 3
ROBERTA_MODEL  = "w11wo/indonesian-roberta-base-sentiment-classifier"

LABEL_MAP = {
    "positive": "positive", "neutral": "neutral", "negative": "negative",
    "POSITIVE": "positive", "NEUTRAL": "neutral", "NEGATIVE": "negative",
    "LABEL_0" : "negative", "LABEL_1": "neutral", "LABEL_2" : "positive",
}
VALID_LABELS = ["positive", "negative", "neutral"]

print(f"Input          : {INPUT_PATH}")
print(f"Output audit   : {VALIDATED_PATH}")
print(f"Output NB03    : {LABELED_PATH}")
print(f"Model          : {ROBERTA_MODEL}")


## 2. Load Kamus

Lima file kamus custom domain MBG digabungkan dengan kamus Nasal (colloquial Indonesian lexicon). Kamus custom melakukan override terhadap Nasal untuk term spesifik domain.


In [ ]:
def load_dict_csv(filename, required_cols=None):
    path = os.path.join(PATH_KAMUS, filename)
    if not os.path.exists(path):
        print(f"  File tidak ditemukan: {filename}")
        return pd.DataFrame(columns=required_cols or ["original_term", "replacement"])
    return pd.read_csv(path, on_bad_lines="skip")


# -- Muat kamus custom ---------------------------------------------------------
df_demoji = load_dict_csv("demoji_code_mbg.csv")
df_akun   = load_dict_csv("akun_x_mbg.csv")
df_alay   = load_dict_csv("kamus_alay_mbg.csv")
df_wl     = load_dict_csv("whitelist_hashtag_mbg.csv")

# -- Muat kamus Nasal (colloquial Indonesian lexicon) --------------------------
# Prioritas file lokal; fallback download (butuh Settings > Internet: On)
_nasal_local = os.path.join(PATH_KAMUS, "colloquial-indonesian-lexicon.csv")
_nasal_url   = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
df_nasal = pd.read_csv(_nasal_local if os.path.exists(_nasal_local) else _nasal_url)

# -- Konversi ke dict ----------------------------------------------------------
whitelist_hashtags = set(
    df_wl["hashtag"].astype(str).str.lower().str.strip()
) if not df_wl.empty else set()

dict_demoji = {
    str(k).strip(): str(v).strip()
    for k, v in zip(df_demoji.get("original_term", []), df_demoji.get("replacement", []))
    if pd.notna(k) and pd.notna(v) and str(k).strip() != ""
}

dict_akun = {
    str(k).strip(): str(v).strip()
    for k, v in zip(df_akun.get("original_term", []), df_akun.get("replacement", []))
    if pd.notna(k) and pd.notna(v)
}

# Merge kamus alay: Nasal sebagai base, custom sebagai override
full_dict_alay = {
    str(k).lower(): str(v).lower()
    for k, v in zip(df_nasal["slang"], df_nasal["formal"])
    if pd.notna(k) and pd.notna(v)
}
dict_custom = {
    str(k).lower(): str(v).lower()
    for k, v in zip(df_alay.get("original_term", []), df_alay.get("replacement", []))
    if pd.notna(k) and pd.notna(v)
}
full_dict_alay.update(dict_custom)

# Singkatan yang tidak boleh dinormalisasi (kata pendek yang bermakna)
protected_words = ["sma", "smp", "sd", "tk", "pt"]
for word in protected_words:
    full_dict_alay.pop(word, None)

# Pisahkan frasa (multi-kata) dan kata tunggal untuk efisiensi replace
dict_phrases   = {k: v for k, v in full_dict_alay.items() if " " in k}
dict_words     = {k: v for k, v in full_dict_alay.items() if " " not in k}
sorted_phrases = sorted(dict_phrases.keys(), key=len, reverse=True)

print(f"Emoji mapping     : {len(dict_demoji):,} entri")
print(f"Akun mapping      : {len(dict_akun):,} entri")
print(f"Whitelist hashtag : {len(whitelist_hashtags):,} tag")
print(f"Kamus alay total  : {len(full_dict_alay):,} ({len(dict_phrases):,} frasa, {len(dict_words):,} kata)")


## 3. Fungsi Preprocessing

Pipeline 4 tahap: case folding, cleaning, tokenization, normalization. Output `text_bert` digunakan untuk IndoRoBERTa (notebook ini) dan IndoBERT+CNN (NB03).

Stemming dan stopword removal tidak diterapkan karena IndoBERT dilatih pada teks natural dan penghapusan kata negasi dapat membalik sentimen.


In [ ]:
def step1_casefolding(text):
    """Lowercase dan decode HTML entities umum di Twitter."""
    if pd.isna(text) or str(text).strip() == "": return ""
    temp = str(text)
    temp = temp.replace("&amp;",  " dan ")
    temp = temp.replace("&lt;",   " kurang dari ")
    temp = temp.replace("&gt;",   " lebih dari ")
    temp = temp.replace("&quot;", '"')
    temp = temp.replace("&#39;",  "'")
    return temp.lower()


def step2_cleaning(text):
    """Cleaning menyeluruh sambil mempertahankan sinyal sentimen."""
    if not text: return ""
    temp = text

    # 2.1 Emotikon ASCII
    temp = re.sub(r"[:;=x][\-~]?[)d\]]+", " senang ", temp)
    temp = re.sub(r"[:;=x][\-~]?[(\[]+",  " sedih ",  temp)

    # 2.2 URL
    temp = re.sub(r"http\S+|www\S+|https\S+", " ", temp, flags=re.MULTILINE)

    # 2.3 Singkatan domain dalam kurung (tidak informatif)
    for pat in [r"\(\s*mbg\s*\)", r"\(\s*bgn\s*\)",
                r"\(\s*apbn\s*\)", r"\(\s*as\s*\)"]:
        temp = re.sub(pat, " ", temp)

    # 2.4 Whitespace dan newline
    temp = temp.replace("\n", " . ")
    temp = re.sub(r"[\t\xa0]", " ", temp)
    temp = re.sub(r"([?!,.])(?=[a-z#])", r"\1 ", temp)

    # 2.5 Suffix slang possesif (otak'nya -> otaknya)
    temp = re.sub(r"(?<=[a-z])['`]+[ya]\b", "nya", temp)

    # 2.6 Penyelamat huruf tunggal (s-nya -> huruf s nya)
    temp = re.sub(r"\b([a-z])(?:-|\'|\s)*nya\b", r"huruf \1 nya", temp)

    # 2.7 Repetisi huruf berlebih (enaaaak -> enak)
    temp = re.sub(r"([a-z])\1{2,}", r"\1", temp)

    # 2.8 Hashtag: whitelist dipertahankan, lainnya dihapus
    for tag in re.findall(r"#\w+", temp):
        clean_tag = tag.lower().lstrip("#")
        if clean_tag in whitelist_hashtags or tag.lower() in whitelist_hashtags:
            temp = temp.replace(tag, " " + tag[1:] + " ")
        else:
            temp = temp.replace(tag, " ")

    # 2.9 Mention: penting -> nama asli (kamus), sisanya dihapus
    for k, v in dict_akun.items():
        temp = re.sub(re.escape(k), " " + v + " ", temp, flags=re.IGNORECASE)
    temp = re.sub(r"@\w+", " ", temp)

    # 2.10 Emoji -> frasa
    for k, v in dict_demoji.items():
        temp = temp.replace(k, " " + v + " ")

    # 2.11 Kata ulang angka (anak2 -> anak anak)
    temp = re.sub(
        r"\b([a-z]{2,})2(nya|ny|ku|mu|an|in|kan|san|kah|lah|pun|)\b",
        r"\1 \1\2", temp,
    )

    # 2.12 Currency shield (rp 15.000 dipertahankan sebagai satuan)
    temp = re.sub(r"\b(?:rp|rupiah)\s*[.,]?\s*(\d)", r"rp \1", temp)
    def _ubah_mata_uang(m):
        angka  = m.group(1).replace(" ", ",")
        suffix = m.group(2) or ""
        return f" rp {angka}{suffix} "
    temp = re.sub(
        r"rp\s*([\d\s.,]+)\s*(k|jt|m|ribu|juta|miliar|)\b",
        _ubah_mata_uang, temp,
    )

    # 2.13 Angka + persen
    temp = re.sub(r"(\d+)\s*%", r"\1 persen", temp)

    # 2.14 Pecahan (1/2 -> setengah)
    temp = re.sub(r"\b1\s*/\s*2\b", " setengah ", temp)
    temp = re.sub(r"\b1\s*/\s*4\b", " seperempat ", temp)
    temp = re.sub(r"\b3\s*/\s*4\b", " tigaperempat ", temp)

    # 2.15 Tanggal dan waktu
    _bulan = "(?:jan|feb|mar|apr|mei|jun|jul|agu|sep|okt|nov|des|" \
             "januari|februari|maret|april|juni|juli|agustus|september|oktober|november|desember)"
    temp = re.sub(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", " ", temp)
    temp = re.sub(rf"\b\d{{1,2}}\s+{_bulan}\s+\d{{2,4}}\b", " tanggal ", temp)
    temp = re.sub(r"\b\d{1,2}[:.\-]\d{2}(?:[:.\-]\d{2})?\b", " ", temp)

    # 2.16 Karakter non-alfanumerik selain tanda baca dasar
    temp = re.sub(r"[^a-z0-9\s\?\!\,\.]", " ", temp)

    # 2.17 Normalisasi tanda baca berulang
    temp = re.sub(r"([?!,.])\1+", r"\1", temp)
    temp = re.sub(r"\s+", " ", temp).strip()
    return temp


def step3_tokenization(text):
    """Whitespace split sebagai parsing helper untuk step berikutnya."""
    return text.split() if text else []


def step4_normalization(tokens):
    """Normalisasi slang (frasa dan kata), negasi, dan dedup frasa."""
    if not tokens: return ""
    temp = " ".join(tokens)

    # 4.1 Frasa multi-kata (urut dari terpanjang)
    for phrase in sorted_phrases:
        if phrase in temp:
            temp = re.sub(rf"\b{re.escape(phrase)}\b", dict_phrases[phrase], temp)

    # 4.2 Kata tunggal
    words = temp.split()
    words = [dict_words.get(w, w) for w in words]
    temp = " ".join(words)

    # 4.3 Negasi umum (nggak/ngga/gak/ga -> tidak)
    temp = re.sub(r"\bnggak\b", "tidak", temp)
    temp = re.sub(r"\bngga\b",  "tidak", temp)
    temp = re.sub(r"\bgak\b",   "tidak", temp)
    temp = re.sub(r"\bga\b",    "tidak", temp)

    # 4.4 Dedup frasa berulang (misal "makan bergizi makan bergizi")
    temp = re.sub(r"\b(\w+(?:\s+\w+){0,2})\s+\1\b", r"\1", temp)

    return re.sub(r"\s+", " ", temp).strip()


def preprocess(text):
    """Pipeline lengkap Step 1-4. Return text_bert siap untuk IndoRoBERTa dan IndoBERT+CNN."""
    s1 = step1_casefolding(text)
    s2 = step2_cleaning(s1)
    s3 = step3_tokenization(s2)
    s4 = step4_normalization(s3)
    return s4


print("Pipeline preprocessing siap.")


## 4. Pengujian Pipeline

Validasi 5 kasus representatif untuk memastikan edge case tertangani dengan benar.


In [ ]:
test_cases = [
    # HTML entity, persen, repetisi huruf, emoji ASCII
    "Anggaran MBG &amp; bansos cair barengan? &lt; 50% yg sampai desa. Keren bangeeeetttt =D\n\n Berdoa saja semoga Allah melingungi kita.",
    # Angka uang, URL, hashtag MBG, format tanggal
    "program makan bergizi gratis (mbg) ini anggarannya rp.15.000 lho di tanggal 17/04/2025\nkeren banget...indonesia.#mbg",
    # Pecahan, negasi, sarkasme
    "hampir 50% dana dipotong? gila aja! 1/2 nya dikorupsi kali ya?!",
    # Negasi ga/gak
    "makanannya ga enak sama sekali, gak layak buat anak sekolah",
    # Mention penting, kata ulang angka
    "@prabowo bilang program ini bagus2 tapi kenyataannya gimana? kita sma sama berusaha, tetap banyak anak SMA yang keracunan.",
]

for i, tc in enumerate(test_cases, 1):
    result = preprocess(tc)
    print(f"Kasus {i}")
    orig_preview = tc if len(tc) <= 100 else tc[:100] + "..."
    print(f"  ORIGINAL  : {orig_preview}")
    print(f"  TEXT_BERT : {result}")
    print(f"  Jumlah kata: {len(result.split())}")
    print("-" * 65)


## 5. Terapkan Preprocessing ke Dataset

Menerapkan pipeline ke seluruh tweet, kemudian post-filter untuk membuang tweet kosong, terlalu pendek, atau duplikat setelah preprocessing.


In [ ]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
n_raw = len(df)
print(f"Data loaded: {n_raw:,} baris")

assert "full_text" in df.columns, "Kolom 'full_text' tidak ditemukan."

# -- Terapkan preprocessing ----------------------------------------------------
print("\nMenerapkan preprocessing Step 1-4...")
df["text_bert"] = df["full_text"].progress_apply(preprocess)

# -- Post-filtering ------------------------------------------------------------
n_before = len(df)
df["bert_word_count"] = df["text_bert"].str.split().str.len().fillna(0).astype(int)

df = df[df["text_bert"].str.strip() != ""].copy()
df = df[df["bert_word_count"] > MIN_WORD_COUNT].copy()
df = df.drop_duplicates(subset=["text_bert"], keep="first").copy()
df = df.reset_index(drop=True)
n_after = len(df)

print(f"\nSebelum post-filtering : {n_before:,}")
print(f"Setelah post-filtering : {n_after:,}  (-{n_before - n_after:,})")

stats = df["bert_word_count"].describe()
print(f"\nStatistik panjang text_bert:")
print(f"  Mean   : {stats['mean']:.2f} kata")
print(f"  Median : {stats['50%']:.0f} kata")
print(f"  Min    : {stats['min']:.0f} kata")
print(f"  Max    : {stats['max']:.0f} kata")

print(f"\nContoh hasil preprocessing:")
display(df[["full_text", "text_bert", "bert_word_count"]].head(3))


## 6. Pelabelan Awal dengan IndoRoBERTa

Model `w11wo/indonesian-roberta-base-sentiment-classifier` digunakan sebagai pseudo-annotator untuk perhitungan Cohen's Kappa. Ground truth penelitian tetap mengacu pada anotasi manual di kolom `label`.


In [ ]:
print(f"Memuat model: {ROBERTA_MODEL}")

tokenizer_rob = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
model_rob     = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL)

sentiment_pipe = pipeline(
    task       = "sentiment-analysis",
    model      = model_rob,
    tokenizer  = tokenizer_rob,
    device     = DEVICE,
    max_length = MAX_LENGTH,
    truncation = True,
    padding    = True,
)

print(f"Label model : {model_rob.config.id2label}")
print(f"Device      : {'GPU' if DEVICE == 0 else 'CPU'}")


In [ ]:
def run_inference_batched(texts, batch_size=BATCH_SIZE):
    """Inference IndoRoBERTa dalam batch dengan fallback per-item."""
    all_results = []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    t_start = time.time()

    for i in tqdm(range(0, len(texts), batch_size),
                  total=total_batches,
                  desc="Inference IndoRoBERTa",
                  bar_format="{l_bar}{bar:25}{r_bar}"):
        batch = texts[i:i + batch_size].tolist()
        try:
            preds = sentiment_pipe(batch)
            all_results.extend(preds)
        except Exception:
            # Fallback: proses per-item jika batch gagal
            for text in batch:
                try:
                    all_results.append(sentiment_pipe([text])[0])
                except Exception:
                    all_results.append({"label": "neutral", "score": 0.0})

    elapsed = time.time() - t_start
    print(f"Inference selesai: {elapsed/60:.1f} menit ({len(texts):,} tweet)")
    return all_results


predictions = run_inference_batched(df["text_bert"])

df["label_roberta_raw"]  = [p["label"] for p in predictions]
df["confidence_roberta"] = [round(p["score"], 4) for p in predictions]
df["label_roberta"]      = df["label_roberta_raw"].map(LABEL_MAP).fillna("neutral")

print(f"\nDistribusi label IndoRoBERTa:")
for lbl, cnt in df["label_roberta"].value_counts().items():
    print(f"  {lbl:10s}: {cnt:,} ({cnt/len(df)*100:.1f}%)")
print(f"\nStatistik confidence score:")
print(df["confidence_roberta"].describe().round(4))


## 7. Distribusi Confidence Score

Visualisasi distribusi confidence untuk memahami tingkat kepastian model. Tweet dengan confidence rendah (<0.60) merupakan kasus ambigu yang perlu prioritas review saat validasi manual.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi confidence keseluruhan
axes[0].hist(df["confidence_roberta"], bins=40,
             color="#534AB7", edgecolor="none", alpha=0.85)
axes[0].axvline(x=0.6, color="#E24B4A", linestyle="--",
                linewidth=1.5, label="Threshold 0.60")
axes[0].axvline(x=0.8, color="#1D9E75", linestyle="--",
                linewidth=1.5, label="Threshold 0.80")
axes[0].set_xlabel("Confidence score", fontsize=11)
axes[0].set_ylabel("Jumlah tweet", fontsize=11)
axes[0].set_title("Distribusi Confidence Score IndoRoBERTa", fontsize=12)
axes[0].legend(fontsize=10)
sns.despine(ax=axes[0])

# Distribusi label per confidence tier
df["conf_tier"] = pd.cut(
    df["confidence_roberta"],
    bins=[0, 0.6, 0.8, 1.0],
    labels=["Rendah (<0.60)", "Sedang (0.60-0.80)", "Tinggi (>0.80)"],
)
tier_dist = df.groupby(["conf_tier", "label_roberta"]).size().unstack(fill_value=0)
tier_dist.plot(kind="bar", ax=axes[1],
               color=["#534AB7", "#E24B4A", "#1D9E75"],
               edgecolor="none", width=0.65)
axes[1].set_xlabel("Confidence tier", fontsize=11)
axes[1].set_ylabel("Jumlah tweet", fontsize=11)
axes[1].set_title("Distribusi Label per Confidence Tier", fontsize=12)
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(title="Label", fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

n_low  = (df["confidence_roberta"] < 0.6).sum()
n_mid  = ((df["confidence_roberta"] >= 0.6) & (df["confidence_roberta"] < 0.8)).sum()
n_high = (df["confidence_roberta"] >= 0.8).sum()
print(f"Confidence rendah (<0.60)   : {n_low:,} ({n_low/len(df)*100:.1f}%)  [prioritas review]")
print(f"Confidence sedang (0.60-0.80): {n_mid:,} ({n_mid/len(df)*100:.1f}%)")
print(f"Confidence tinggi (>=0.80)  : {n_high:,} ({n_high/len(df)*100:.1f}%)")


## 8. Export Output

Dua file disimpan: `final_validated_mbg.csv` sebagai audit trail penuh (semua kolom, semua baris) dan `final_mbg_labeled.csv` untuk konsumsi NB03 (hanya baris berlabel valid, kolom esensial).


In [ ]:
# Pastikan kolom label ada (input dari anotasi manual)
if "label" not in df.columns:
    df["label"] = ""
if "annotator_notes" not in df.columns:
    df["annotator_notes"] = ""

# Drop kolom helper yang tidak diperlukan di output
df = df.drop(columns=["label_roberta_raw", "conf_tier"], errors="ignore")

# -- Output 1: final_validated_mbg.csv (audit trail penuh) ---------------------
validated_cols = [
    "id_str", "created_at", "period",
    "full_text", "text_bert", "bert_word_count",
    "label", "label_roberta", "confidence_roberta", "annotator_notes",
    "username", "source_keyword", "lang",
]
validated_cols = [c for c in validated_cols if c in df.columns]
df_validated = df[validated_cols].copy()
df_validated.to_csv(VALIDATED_PATH, index=False, encoding="utf-8-sig")

# -- Output 2: final_mbg_labeled.csv (untuk NB03) ------------------------------
# Hanya baris dengan label manual valid, kolom esensial untuk training
labeled_cols = [
    "id_str", "created_at", "period",
    "full_text", "text_bert", "bert_word_count",
    "label", "source_keyword",
]
labeled_cols = [c for c in labeled_cols if c in df.columns]
df_labeled = df[df["label"].isin(VALID_LABELS)][labeled_cols].copy()
df_labeled.to_csv(LABELED_PATH, index=False, encoding="utf-8-sig")

print("=== EXPORT ===")
print(f"Audit trail    : {VALIDATED_PATH}")
print(f"  Shape        : {df_validated.shape[0]:,} baris x {df_validated.shape[1]} kolom")
print(f"Input NB03     : {LABELED_PATH}")
print(f"  Shape        : {df_labeled.shape[0]:,} baris x {df_labeled.shape[1]} kolom")
print(f"  Label valid  : {df_labeled['label'].value_counts().to_dict()}")


## 9. Inter-Annotator Agreement (Cohen's Kappa)

Menghitung Cohen's Kappa antara label manual peneliti dan prediksi IndoRoBERTa. Interpretasi mengikuti Landis dan Koch (1977): kappa >= 0.61 = substantial agreement (target minimum penelitian).


In [ ]:
def compute_iaa(filepath=None):
    """Hitung Cohen's Kappa dari file audit trail."""
    if filepath is None:
        filepath = VALIDATED_PATH
    df_labeled = pd.read_csv(filepath, low_memory=False)

    df_valid = df_labeled[
        df_labeled["label"].isin(VALID_LABELS) &
        df_labeled["label_roberta"].isin(VALID_LABELS)
    ].copy()

    total         = len(df_labeled)
    total_labeled = len(df_labeled[df_labeled["label"].isin(VALID_LABELS)])
    total_valid   = len(df_valid)

    if total_valid == 0:
        print("Belum ada label manual valid. Isi kolom 'label' terlebih dahulu.")
        return None

    kappa = cohen_kappa_score(df_valid["label"], df_valid["label_roberta"])

    if kappa < 0.20:   interp = "Slight"
    elif kappa < 0.40: interp = "Fair"
    elif kappa < 0.60: interp = "Moderate"
    elif kappa < 0.80: interp = "Substantial"
    else:              interp = "Almost Perfect"

    print("=== INTER-ANNOTATOR AGREEMENT ===")
    print(f"Total data           : {total:,}")
    print(f"Sudah dilabeli manual: {total_labeled:,} ({total_labeled/total*100:.1f}%)")
    print(f"Valid untuk IAA      : {total_valid:,}")
    print(f"Cohen's Kappa        : {kappa:.4f}")
    print(f"Interpretasi         : {interp}")
    print(f"Target (>= 0.61)     : {'TERPENUHI' if kappa >= 0.61 else 'BELUM TERPENUHI'}")

    # -- Confusion matrix dan distribusi label --------------------------------
    cm = confusion_matrix(
        df_valid["label"], df_valid["label_roberta"], labels=VALID_LABELS
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=VALID_LABELS, yticklabels=VALID_LABELS,
                ax=axes[0])
    axes[0].set_xlabel("IndoRoBERTa (pseudo-annotator)", fontsize=11)
    axes[0].set_ylabel("Manual (anotator manusia)", fontsize=11)
    axes[0].set_title(f"Confusion Matrix IAA (kappa = {kappa:.4f})", fontsize=12)

    label_counts = (df_labeled[df_labeled["label"].isin(VALID_LABELS)]
                    ["label"].value_counts()
                    .reindex(VALID_LABELS, fill_value=0))
    colors = ["#534AB7", "#E24B4A", "#1D9E75"]
    bars = axes[1].bar(label_counts.index, label_counts.values,
                       color=colors, edgecolor="none", alpha=0.85)
    for bar, val in zip(bars, label_counts.values):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 10,
                     f"{val:,}", ha="center", va="bottom", fontsize=9)
    axes[1].set_xlabel("Label manual", fontsize=11)
    axes[1].set_ylabel("Jumlah tweet", fontsize=11)
    axes[1].set_title("Distribusi Label Manual", fontsize=12)
    sns.despine(ax=axes[1])

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/iaa_result.png", dpi=150, bbox_inches="tight")
    plt.show()

    return kappa


kappa = compute_iaa()
